# 30 · Multimodal comparison — fMRI vs EEG
Compares the two lines with the same protocol (same CLIP target, same correct/permuted/zero controls, same frozen SD generator). The datasets differ (NSD scenes vs THINGS objects), so compare each modality against **its own chance level**, not against each other in absolute terms.

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.utils import load_json
import pandas as pd

# Point these at the retrieval-ablation outputs of each line (Experiment 2).
RUNS = {
    'fMRI (NSD)':  'outputs/exp02_retrieval_ablation',
    'EEG (THINGS)':'outputs/exp02_eeg_retrieval_ablation',
}

## Retrieval by condition, per modality
Reads each experiment's tidy `metrics/summary_table.csv`.

In [ ]:
frames = []
for name, d in RUNS.items():
    p = Path(d)/'metrics'/'summary_table.csv'
    if not p.exists():
        print('missing:', p, '(run Experiment 2 for this modality first)'); continue
    df = pd.read_csv(p)
    df = df[(df.subject_id=='all') & (df.metric_name=='retrieval/top5')]
    for _, r in df.iterrows():
        frames.append({'modality': name, 'condition': r['condition'], 'top5': r['value']})
tab = pd.DataFrame(frames)
tab.pivot_table(index='modality', columns='condition', values='top5') if len(tab) else 'no runs found'

In [ ]:
if len(tab):
    piv = tab.pivot_table(index='condition', columns='modality', values='top5')
    piv.plot(kind='bar', figsize=(8,4)); plt.ylabel('Top-5'); plt.title('Top-5 retrieval by condition'); plt.show()

## Verdict per modality

In [ ]:
from src.evaluation import conclusion_from_summary
for name, d in RUNS.items():
    p = Path(d)/'metrics'/'summary_table.csv'
    if not p.exists(): continue
    s = pd.read_csv(p)
    c = conclusion_from_summary(s, metric='retrieval/top5', subject='all')
    print(f"{name:>14}: {c['decision']}")

**Takeaway:** report each modality's `correct` vs controls separately. A modality is only credited with visual decoding if `correct ≫ permuted ≈ zero` for its own data.